In [0]:
import os
import time
import json
import pandas as pd
from datetime import datetime

#notebook_dir = "/Workspace/Users/cnzhoufang@qq.com/Databricks-Test/aws/3.5/"
#notebook_dir = "/Workspace/Users/cnzhoufang@qq.com/Databricks-Test/aws/3.5+photon/"
notebook_dir = "/Workspace/Users/cnzhoufang@qq.com/Databricks-Test/aws/4.0_24/"
#notebook_dir = "/Workspace/Users/cnzhoufang@qq.com/Databricks-Test/aws/4.0+photon/"
#notebook_dir = "/Workspace/Users/cnzhoufang@qq.com/Databricks-Test/aws/test/"

    
# 2. 定义执行顺序 (只需填写文件名，不带后缀)
order = ["1","2","3","4","5","6","7","8","9",
         "10","11","12","13","14a","14b","15","16","17","18","19",
         "20","21","22","23a","23b","24a","24b","25","26","27","28","29",
         "30","31","32","33","34","35","36","37","38","39a","39b",
         "40","41","42","43","44","45","46","47","48","49",
         "50","51","52","53","54","55","56","57","58","59",
         "60","61","62","63","64","65","66","67","68","69",
         "70","71","72","73","74","75","76","77","78","79",
         "80","81","82","83","84","85","86","87","88","89",
         "90","91","92","93","94","95","96","97","98","99"]

# 3. 根据给定顺序构建文件列表
files = []
for name in order:
    # 优先找 .ipynb，找不到再找 .sql
    ipynb_file = f"{name}.ipynb"
    sql_file = f"{name}.sql"
    
    if os.path.exists(os.path.join(notebook_dir, ipynb_file)):
        files.append(ipynb_file)
    elif os.path.exists(os.path.join(notebook_dir, sql_file)):
        files.append(sql_file)
    else:
        print(f"警告: 未找到文件 {name}.ipynb 已跳过。")

# 用于存储执行结果的列表
results = []

# 开始按照 order 顺序遍历执行
for file in files:
    file_path = os.path.join(notebook_dir, file)
    print(f"\n处理文件: {file}")
    print("=" * 80)

    if file.endswith(".ipynb"):
        # 解析 .ipynb 文件的 JSON 格式
        with open(file_path, "r", encoding="utf-8") as f:
            notebook_content = json.load(f)
        
        # 获取 notebook 级别的语言设置
        notebook_metadata = notebook_content.get("metadata", {})
        databricks_metadata = notebook_metadata.get("application/vnd.databricks.v1+notebook", {})
        notebook_language = databricks_metadata.get("language", "")
        
        # 从 notebook 中提取 SQL 代码单元
        cells = notebook_content.get("cells", [])
        sql_cell_count = 0
        
        for idx, cell in enumerate(cells):
            cell_type = cell.get("cell_type", "")
            source = cell.get("source", [])
            
            # 将 source 转换为字符串（可能是列表或字符串）
            if isinstance(source, list):
                code = "".join(source).strip()
            else:
                code = source.strip()
            
            # 检查是否是 SQL 代码
            metadata = cell.get("metadata", {})
            cell_language = metadata.get("language", "")
            
            is_sql = False
            
            # 1. 检查 cell 级别的语言标识
            if cell_language == "sql" or cell_type == "sql":
                is_sql = True
            # 2. 检查 notebook 级别的语言标识（如果 cell 是 code 类型）
            elif cell_type == "code" and notebook_language == "sql":
                is_sql = True
            # 3. 检查代码是否以 %sql 或 %%sql 开头
            elif code.startswith("%sql") or code.startswith("%%sql"):
                is_sql = True
                # 移除 magic command
                code = code.replace("%sql", "", 1).replace("%%sql", "", 1).strip()
            
            if is_sql and code:
                sql_cell_count += 1
                print(f"\n找到 SQL Cell {idx+1}:")
                print(f"代码内容:\n{code}")
                print("-" * 40)
                
                try:
                    spark.conf.set("spark.job.description", f"{file}")
                    # 关闭 disk cache, metadata cache, photon cache
                    spark.conf.set("spark.databricks.io.cache.enabled", "false")
                    spark.conf.set("spark.databricks.io.cache.metadata.enabled", "false")
                    spark.conf.set("spark.databricks.photon.io.cache.enabled", "false")
                    spark.conf.set("spark.databricks.photon.io.cache.metadata.enabled", "false")
                    #
                    start_time = time.time()
                    df = spark.sql(code)
                    display(df)
                    execution_time = time.time() - start_time
                    print(f"{file} [Cell {idx+1}]: SQL execution time: {execution_time:.4f} seconds")
                    
                    # 记录结果
                    results.append({
                        "文件名": file, 
                        "Cell编号": idx + 1,
                        "状态": "成功", 
                        "执行时间(秒)": round(execution_time, 4)
                    })
                except Exception as e:
                    print(f"{file} [Cell {idx+1}]: Error - {str(e)}")
                    results.append({
                        "文件名": file,
                        "Cell编号": idx + 1,
                        "状态": f"失败: {str(e)}",
                        "执行时间(秒)": 0
                    })
        
        if sql_cell_count == 0:
            print(f"{file}: No SQL cells found")
    
    print("\n")

# 将结果转换为 DataFrame 并保存为 CSV
if results:
    results_df = pd.DataFrame(results)
    
    # 生成带时间戳的文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"execution_results_{timestamp}.csv"
    csv_path = os.path.join(notebook_dir, csv_filename)
    
    results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    
    print("\n" + "=" * 80)
    print("执行结果汇总:")
    print("=" * 80)
    display(results_df)
    print(f"\n结果已保存到: {csv_path}")
else:
    print("\n没有执行任何 SQL 语句")